# Week 12: Practical Use of Language Models

**Foundations of Data Science**

## In this lesson

1. Text as data: the problem is turning words into numbers, and which numbers
2. Bag of words and TF-IDF: the baseline that is hard to beat
3. A text classifier, and reading what it learned
4. Embeddings: dense vectors, similarity, nearest neighbors
5. Modern embeddings from a language model
6. Prompting a language model as a classifier, and measuring it like any other
7. Where language models fit in a data science workflow, and where they don't
8. Pitfalls

## The one idea

**A language model is a model. It gets a held-out test set, a baseline, a confusion matrix, and an error bar, same as everything else in this course.** The temptation with LLMs is to skip evaluation because the output reads well. Reading well is not being right, and the week 8 discipline applies with no exceptions.

The data: 3,746 posts from four Usenet newsgroups (medicine, space, baseball, politics), the standard 20 Newsgroups benchmark. It stands in for the real task, which in a health system is routing free-text messages (patient portal messages, referral notes, incident reports) to the right team. **The decision: given a message, which department gets it, and how much can we trust the routing?** Real patient text can't leave the building, which section 7 takes seriously.

## Setup

<!-- nb-links -->
[Download this notebook](week_12_language_models.ipynb?download=1){download="week_12_language_models.ipynb"} &nbsp;|&nbsp; [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lshpaner/fds_course_students/blob/main/notebooks/week_12/week_12_language_models.ipynb)


In [ ]:
# --- Setup: works in Colab and locally ---------------------------------------
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "fds_course"

if IN_COLAB:
    # Clone the course repo and move into it (skip if already there)
    if not Path(REPO).exists():
        !git clone -q https://github.com/lshpaner/{REPO}.git
    os.chdir(REPO)
    !pip install -q -r requirements.txt
else:
    # Local: the notebook lives in notebooks/week_XX/, so go up two levels to the repo root
    if Path.cwd().name.startswith("week_"):
        os.chdir(Path.cwd().parents[1])

# Make core/ importable
sys.path.insert(0, os.getcwd())

# Fetch this week's raw data only if it isn't already on disk, then build the processed table
if not Path("data/raw/newsgroups.parquet").exists():
    !python data_gen.py fetch --name newsgroups

print("Working directory:", os.getcwd())

## 1. Look at the text

Same first-contact discipline as week 1: shape, a few rows, and the things specific to text: length distribution, class balance, and what the raw strings actually look like (headers, quotes, and signatures were already stripped by the loader; real data won't be that kind).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

docs = pd.read_parquet("data/raw/newsgroups.parquet")
docs["n_words"] = docs["text"].str.split().str.len()
docs = docs[docs["n_words"] >= 5].reset_index(drop=True)      # drop empty and near-empty posts

print(docs.shape)
print(docs["category"].value_counts().to_dict())
print("\nwords per post:", docs["n_words"].describe().round(0).to_dict())
print("\n--- example (sci.med) ---\n", docs.loc[docs["category"] == "sci.med", "text"].iloc[26][:600])

## 2. Bag of words and TF-IDF

The oldest trick still works: count how often each word appears in each document, and treat the counts as features. A document becomes a very long, very sparse vector (one entry per word in the vocabulary). Word order is thrown away, hence "bag."

**TF-IDF** rescales the counts. Term frequency (how often the word appears here) times inverse document frequency (how rare the word is across all documents). "The" appears everywhere, so its IDF is near zero; "orbit" appears in a few hundred posts, mostly about space, so it gets a high weight. The result is that each document is described by the words that distinguish it.

Then any classifier from week 7 works on those vectors. Logistic regression is the standard choice: fast, well calibrated, and the coefficients are readable.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

X_train, X_test, y_train, y_test = train_test_split(docs["text"], docs["category"], test_size=0.25, stratify=docs["category"], random_state=42)

tfidf_lr = Pipeline([
    ("tfidf", TfidfVectorizer(min_df=3, max_df=0.5, ngram_range=(1, 2), sublinear_tf=True, stop_words="english")),
    ("lr", LogisticRegression(C=5, max_iter=2000)),
]).fit(X_train, y_train)

pred = tfidf_lr.predict(X_test)
print("vocabulary size:", len(tfidf_lr["tfidf"].vocabulary_))
print("test accuracy:", round(accuracy_score(y_test, pred), 3), "   (chance is 0.25; majority class is", round(y_test.value_counts(normalize=True).max(), 2), ")")
print(classification_report(y_test, pred, digits=3))

In [ ]:
import seaborn as sns
labels = sorted(docs["category"].unique())
cm = confusion_matrix(y_test, pred, labels=labels)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_xlabel("predicted"); ax.set_ylabel("actual"); ax.set_title("TF-IDF + logistic regression")
plt.show()

About 89% accuracy from word counts and a linear model, with the confusions where you'd expect (medicine and politics share vocabulary about policy and risk; space and medicine share science-writing vocabulary). This is the baseline. Anything fancier has to beat it on the same test set, and the cost of the fancier thing has to be worth the gap.

## 3. Reading what it learned

A linear model on TF-IDF features has one coefficient per word per class. The largest ones are the words the model uses to recognize each category. Read them: they tell you whether the model learned the topic or learned an artifact (a signature, a frequent poster's name, a formatting quirk).

In [ ]:
vocab = np.array(tfidf_lr["tfidf"].get_feature_names_out())
coef = tfidf_lr["lr"].coef_
top = pd.DataFrame({cls: vocab[np.argsort(coef[i])[-12:]][::-1] for i, cls in enumerate(tfidf_lr["lr"].classes_)})
top

Mostly topic words: `orbit`, `shuttle`, `treatment`, `stadium`. Now look at the politics column: `clinton`, `koresh`, `batf`. Those are not "politics"; they are *spring 1993* (the Waco siege). And `jewish` in the baseball column is one argumentative thread, not the sport. The model learned the moment and the quirks of this corpus alongside the topic, and would misfire on political posts from any other year. That is the text version of leakage, and this table is how you catch it: read the top words for every class, every time, and ask whether each one would still be a signal on next year's data.

## 4. Embeddings

TF-IDF vectors have tens of thousands of dimensions and treat "physician" and "doctor" as unrelated. An **embedding** is a short dense vector (a few hundred numbers) where similar meanings land near each other. The oldest way to get one from a corpus is **latent semantic analysis**: run PCA (week 9) on the TF-IDF matrix. Words that co-occur across documents collapse into shared directions.

Similarity between two vectors is measured by **cosine similarity**: the cosine of the angle between them, 1 for identical direction, 0 for unrelated. With embeddings in hand, "find the five most similar documents" is a nearest-neighbor lookup, which is the operation behind search, recommendation, and retrieval-augmented generation.

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

tfidf = tfidf_lr["tfidf"]
X_train_tfidf = tfidf.transform(X_train)
lsa = TruncatedSVD(n_components=200, random_state=42).fit(X_train_tfidf)
E_train = lsa.transform(X_train_tfidf)
print("embedding shape:", E_train.shape, "  variance kept:", round(lsa.explained_variance_ratio_.sum(), 2))

# Nearest neighbors of one test post
query = X_test.iloc[0]
q = lsa.transform(tfidf.transform([query]))
sims = cosine_similarity(q, E_train)[0]
nn = np.argsort(sims)[-5:][::-1]
print("\nquery (", y_test.iloc[0], "):", query[:200].replace("\n", " "), "...")
for i in nn:
    print(f"\n  sim {sims[i]:.2f}  [{y_train.iloc[i]}]  {X_train.iloc[i][:120].replace(chr(10), ' ')} ...")

In [ ]:
# Embeddings as features: does a 200-dim LSA vector classify as well as 30,000 TF-IDF dimensions?
lsa_lr = LogisticRegression(C=5, max_iter=2000).fit(E_train, y_train)
E_test = lsa.transform(tfidf.transform(X_test))
print("LSA (200 dims) + LR test accuracy:", round(accuracy_score(y_test, lsa_lr.predict(E_test)), 3))

## 5. Modern embeddings from a language model

LSA learns from the 2,800 documents in front of it. A pretrained language model has read a large share of the internet, and its embeddings encode meaning it learned elsewhere: "myocardial infarction" sits near "heart attack" even if neither appears in your corpus. For small or specialized datasets that transfer is the whole advantage.

The cell below calls an embedding endpoint through the OpenAI-compatible client, which also works with local servers (Ollama, LM Studio) and most hosted providers by changing `base_url`. It reads credentials from a `.env` file at the repo root (`OPENAI_API_KEY=...`, optionally `OPENAI_BASE_URL=...` and `EMBED_MODEL=...`). **If no key is present it skips**, so the notebook still runs top to bottom; the printed note says what you'd see.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY")
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
CHAT_MODEL = os.getenv("CHAT_MODEL", "gpt-4o-mini")

if not API_KEY:
    print("No OPENAI_API_KEY in .env: skipping the language-model embedding cell.")
    print("With a key, this cell embeds the training and test posts (~3,500 calls batched 100 at a time, a few cents),")
    print("fits the same logistic regression on the vectors, and typically lands 1 to 3 points above the TF-IDF baseline.")
else:
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY, base_url=os.getenv("OPENAI_BASE_URL"))

    def embed(texts: list[str], batch: int = 100) -> np.ndarray:
        out = []
        for k in range(0, len(texts), batch):
            resp = client.embeddings.create(model=EMBED_MODEL, input=[t[:4000] for t in texts[k:k + batch]])
            out.extend(d.embedding for d in resp.data)
        return np.array(out)

    E_train_lm = embed(X_train.tolist()); E_test_lm = embed(X_test.tolist())
    lm_lr = LogisticRegression(C=5, max_iter=2000).fit(E_train_lm, y_train)
    print("LM embeddings shape:", E_train_lm.shape)
    print("LM embeddings + LR test accuracy:", round(accuracy_score(y_test, lm_lr.predict(E_test_lm)), 3))

## 6. Prompting a language model as a classifier

No training at all: describe the four categories in a prompt, hand the model a post, ask for the label. This is **zero-shot classification**. It's the fastest way to get *a* classifier, and the slowest and most expensive way to run one at scale. It is also a model, so it gets the same treatment: held-out test set, accuracy, confusion matrix, compared to the baseline.

Three things to notice in the code: the prompt pins the output format (one label from a fixed list) so the answer can be scored automatically; temperature is zero so the result is repeatable; and the evaluation is on a random sample of the *same test set* the baseline used, so the comparison is fair.

In [ ]:
if not API_KEY:
    print("No OPENAI_API_KEY in .env: skipping the zero-shot classification cell.")
    print("With a key, this cell classifies 120 held-out posts by prompt alone. Expect accuracy in the high 80s to low 90s,")
    print("below the trained TF-IDF model, with most errors on the medicine/politics boundary, at ~100x the per-document cost.")
else:
    import time
    LABELS = {"sci.med": "medicine and health", "sci.space": "space and astronomy",
              "rec.sport.baseball": "baseball", "talk.politics.misc": "politics"}
    system = ("You classify Usenet posts into exactly one category. Reply with only the category id, nothing else.\n"
              + "\n".join(f"{k}: {v}" for k, v in LABELS.items()))

    sample = X_test.sample(120, random_state=7)
    zs_pred, t0 = [], time.time()
    for text in sample:
        resp = client.chat.completions.create(model=CHAT_MODEL, temperature=0, max_tokens=12,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": text[:3000]}])
        ans = resp.choices[0].message.content.strip()
        zs_pred.append(ans if ans in LABELS else "unparseable")
    zs_pred = pd.Series(zs_pred, index=sample.index)
    print(f"{len(sample)} posts in {time.time() - t0:.0f}s")
    print("zero-shot accuracy:", round(accuracy_score(y_test.loc[sample.index], zs_pred), 3),
          "   TF-IDF on the same posts:", round(accuracy_score(y_test.loc[sample.index], tfidf_lr.predict(sample)), 3))
    print("unparseable replies:", (zs_pred == "unparseable").sum())

## 7. Where language models fit

Honest map of the options for a text task, from a data scientist's seat:

| Approach | Needs | Cost per document | Strength | Weakness |
|---|---|---|---|---|
| TF-IDF + linear model | labeled data (hundreds+) | ~0 | fast, auditable, calibrated, runs anywhere | no transfer; brittle to vocabulary shift |
| LM embeddings + linear model | labeled data, an embedding endpoint | fractions of a cent | transfer from pretraining; strong on small data | depends on an external model; harder to audit |
| Zero/few-shot prompting | a prompt, an LLM endpoint | cents | no labels needed; flexible categories | slow, costly at scale, non-deterministic across model versions, must be re-evaluated on every model change |
| Fine-tuned LLM | thousands of labels, compute | varies | best accuracy on hard tasks | most expensive to build and maintain |

Where LLMs earn their keep in a data science workflow, in rough order of value: **extraction** (pull structured fields out of free text, then do ordinary analysis on the fields), **labeling** (bootstrap a training set a human then checks), **code and documentation** (writing the boilerplate, explaining someone else's), **summarization** of long documents for triage. In every case the output feeds a step that gets measured.

Two rules that don't bend:

1. **Protected text does not leave the building.** Patient messages, clinical notes, anything with PHI or PII goes to a model that runs inside your environment (a local model, or a contracted deployment with a signed agreement), never to a public API. The code above reads a public benchmark; the same code pointed at patient text would be a reportable breach.
2. **Evaluate on held-out data every time the model changes.** A provider updating the model behind an endpoint is a new model. Your week 8 test set is the regression test.

## 8. Pitfalls

- **Contamination.** Public benchmarks (this one included) are in the training data of large models. A zero-shot score on 20 Newsgroups overstates what you'd get on your own text. Evaluate on data the model can't have seen.
- **Prompt as hyperparameter.** Small wording changes move accuracy by points. Tune the prompt on a validation set, then score once on test, exactly like `C` in week 7.
- **Reading fluency as correctness.** The confusion matrix doesn't care how confident the prose sounds.
- **Costs that scale.** A prompt that costs a cent is free at 100 documents and $10,000 at a million. Compute the per-document cost before choosing the approach.
- **Non-determinism.** Set temperature to zero, log the model version, and still expect drift over time.

## Looking ahead
- **Reading**: for next week, Barocas, Hardt & Narayanan, *Fairness and Machine Learning*, chapter 3 (free online); and Obermeyer et al., *Science* 2019, "Dissecting racial bias in an algorithm used to manage the health of populations." The week 7 model gets audited.